In [1]:
%load_ext watermark
import numpy as np
from skimage.io import imread
from pathlib import Path
#from aicspylibczi import CziFile
import os
import tifffile
import itk #pip install itk-elastix(in case itk has an error: ParameterObject not found)

In [2]:
#enter folder with tif files
#enter output folder where images will be saved

input_folder = "/Users/marchenk/Desktop/Karo/tofuse"   # Folder with TIFF files
output_folder = "/Users/marchenk/Desktop/Karo/results"

os.makedirs(output_folder, exist_ok=True)

In [3]:
# -----------------------------
# Helper Functions
# -----------------------------

def parse_filename(fname):
    """Extract position and angle info from filename, e.g., position1-0degree.tif"""
    name_without_ext = os.path.splitext(fname)[0]
    parts = name_without_ext.split('-')
    position_part = next((p for p in parts if 'position' in p.lower()), None)
    ending_part = parts[-1] if parts else None
    return position_part, ending_part

def normalize_tif_shape(tif_path):
    """
    Load TIFF, ensure shape is Z,C,Y,X.
    Handles common shapes:
      - (Y, X) -> (1,1,Y,X)
      - (C, Y, X) -> (1,C,Y,X)
      - (Z, Y, X) -> (Z,1,Y,X)
      - (Z, C, Y, X) -> unchanged
    """
    img = tifffile.imread(tif_path)

    if img.ndim == 2:
        img = img[np.newaxis, np.newaxis, :, :]  # (Y,X) -> (1,1,Y,X)
    elif img.ndim == 3:
        # Assume (C,Y,X) if first dim small (<10), else (Z,Y,X)
        if img.shape[0] < 10:
            img = img[np.newaxis, ...]  # (C,Y,X) -> (1,C,Y,X)
        else:
            img = img[:, np.newaxis, ...]  # (Z,Y,X) -> (Z,1,Y,X)
    elif img.ndim == 4:
        # Assume already Z,C,Y,X
        pass
    else:
        raise ValueError(f"Unexpected image shape: {img.shape}")

    return img.astype(np.float32)

In [4]:
# -----------------------------
# Process Files
# -----------------------------

# Collect TIFF files and group by position
all_files = [f for f in os.listdir(input_folder) if f.endswith('.tif')]
positions = {}
for fname in all_files:
    pos, end = parse_filename(fname)
    if pos is None or end is None:
        print(f"Skipping unrecognized file: {fname}")
        continue
    if pos not in positions:
        positions[pos] = {}
    positions[pos][end] = fname

# -----------------------------
# MAIN LOOP (HEADLESS)
# -----------------------------

for pos, files_dict in positions.items():
    if '0degree' in files_dict and '180degree' in files_dict:
        # TIFF paths
        tif1_path = os.path.join(input_folder, files_dict['0degree'])
        tif2_path = os.path.join(input_folder, files_dict['180degree'])

        # Load and normalize shapes
        image1 = normalize_tif_shape(tif1_path)
        image2 = normalize_tif_shape(tif2_path)

        #print(f"{pos}: image1 shape {image1.shape}, image2 shape {image2.shape}")

        Z, C, Y, X = image1.shape
        fused_image_all_channels = np.zeros_like(image1, dtype=np.float32)

        # -----------------------------
        # PROCESS CHANNELS
        # -----------------------------
        for ch in range(C):
            print(f"Processing channel {ch} for position {pos}")

            img1_ch = image1[:, ch, :, :]
            img2_ch = image2[:, ch, :, :]

            # Rotate image2 by 180 degrees in Z-X plane
            image2_rot = np.rot90(img2_ch, axes=(0, 2), k=2)

            # Normalize intensities
            img1_ch = img1_ch / np.percentile(img1_ch, 99)
            image2_rot = image2_rot / np.percentile(image2_rot, 99)

            # Registration
            parameter_object = itk.ParameterObject.New()
            default_affine_parameter_map = parameter_object.GetDefaultParameterMap('affine', 3)
            default_affine_parameter_map['FinalBSplineInterpolationOrder'] = ['0']
            default_affine_parameter_map['NumberOfResolutions'] = ['4']
            parameter_object.AddParameterMap(default_affine_parameter_map)

            result_image, _ = itk.elastix_registration_method(
                img1_ch,
                image2_rot,
                parameter_object=parameter_object,
                log_to_console=False
            )
            image2_rot_registered = np.asarray(result_image).astype(np.float32)

            # Fusion using gradient magnitude weights
            gradient1 = np.gradient(img1_ch)
            gradient_magnitude1 = np.sqrt(sum(g**2 for g in gradient1))
            gradient2 = np.gradient(image2_rot_registered)
            gradient_magnitude2 = np.sqrt(sum(g**2 for g in gradient2))

            weights1 = gradient_magnitude1 / (gradient_magnitude1 + gradient_magnitude2 + 1e-8)
            weights2 = gradient_magnitude2 / (gradient_magnitude1 + gradient_magnitude2 + 1e-8)

            fused_channel = weights1 * img1_ch + weights2 * image2_rot_registered

            fused_image_all_channels[:, ch, :, :] = fused_channel

        # -----------------------------
        # Save fused image
        # -----------------------------
        output_filename = f"{pos}_fused_all_channels.tif"
        output_path = os.path.join(output_folder, output_filename)
        tifffile.imwrite(output_path, fused_image_all_channels)
        print(f"✅ Saved fused image for {pos} at: {output_path}")

    else:
        print(f"⚠ Missing pair for position {pos}: found files {list(files_dict.keys())}")

Processing channel 0 for position position49
✅ Saved fused image for position49 at: /Users/marchenk/Desktop/Karo/results/position49_fused_all_channels.tif
